In [1]:
# CELL 1 — Read raw clinical encounters JSON

from pyspark.sql.functions import current_timestamp, lit

batch_id = "clinical_encounters_batch_001"
source_file = "clinical_encounters.json"

clinical_encounters_df = (
    spark.read
    .json("Files/raw/clinical_encounters.json")
    .withColumn("batch_id", lit(batch_id))
    .withColumn("source_file_name", lit(source_file))
    .withColumn("ingestion_timestamp", current_timestamp())
)

print("Rows loaded:", clinical_encounters_df.count())
clinical_encounters_df.printSchema()
print(clinical_encounters_df.columns)

display(clinical_encounters_df.limit(20))

StatementMeta(, dc775bdb-7a22-4d21-9d5b-deb145de9eb1, 3, Finished, Available, Finished, False)

Rows loaded: 973589
root
 |-- FACILITY_CITY: string (nullable = true)
 |-- attending_staff_id: string (nullable = true)
 |-- billing_amount: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- encounter_timestamp: string (nullable = true)
 |-- encountertype: string (nullable = true)
 |-- facility_c: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- insurance_claim_status: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- primary_diagnosis_code: string (nullable = true)
 |-- procedure_: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = false)
 |-- source_file_name: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)

['FACILITY_CITY', 'attending_staff_id', 'billing_amount', 'encounter_id', 'encounter_timestamp', 'encountertype', 'facility_c', 'facility_id', 'insurance_claim_status', 'pat

SynapseWidget(Synapse.DataFrame, 090e0db7-15a5-4eb9-83eb-75b630f3d06e)

In [2]:
# CELL 2 — Standardize column names correctly

clinical_encounters_df = clinical_encounters_df.toDF(
    *[
        column.strip().lower()
        for column in clinical_encounters_df.columns
    ]
)

clinical_encounters_df.printSchema()
print(clinical_encounters_df.columns)

StatementMeta(, dc775bdb-7a22-4d21-9d5b-deb145de9eb1, 4, Finished, Available, Finished, False)

root
 |-- facility_city: string (nullable = true)
 |-- attending_staff_id: string (nullable = true)
 |-- billing_amount: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- encounter_timestamp: string (nullable = true)
 |-- encountertype: string (nullable = true)
 |-- facility_c: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- insurance_claim_status: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- primary_diagnosis_code: string (nullable = true)
 |-- procedure_: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = false)
 |-- source_file_name: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)

['facility_city', 'attending_staff_id', 'billing_amount', 'encounter_id', 'encounter_timestamp', 'encountertype', 'facility_c', 'facility_id', 'insurance_claim_status', 'patient_id', 'primary_d

In [3]:
# CELL 3 — Write clinical encounters to Bronze Delta table

spark.sql("DROP TABLE IF EXISTS bronze_clinical_encounters")

(
    clinical_encounters_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_clinical_encounters")
)

print(
    "Bronze rows:",
    spark.table("bronze_clinical_encounters").count()
)

print("bronze_clinical_encounters created successfully.")

StatementMeta(, dc775bdb-7a22-4d21-9d5b-deb145de9eb1, 5, Finished, Available, Finished, False)

Bronze rows: 973589
bronze_clinical_encounters created successfully.


In [4]:
# CELL 4 — Verify Bronze clinical encounters

bronze_clinical_encounters_df = spark.table(
    "bronze_clinical_encounters"
)

bronze_clinical_encounters_df.printSchema()
print(bronze_clinical_encounters_df.columns)
display(bronze_clinical_encounters_df.limit(20))

StatementMeta(, dc775bdb-7a22-4d21-9d5b-deb145de9eb1, 6, Finished, Available, Finished, False)

root
 |-- facility_city: string (nullable = true)
 |-- attending_staff_id: string (nullable = true)
 |-- billing_amount: string (nullable = true)
 |-- encounter_id: string (nullable = true)
 |-- encounter_timestamp: string (nullable = true)
 |-- encountertype: string (nullable = true)
 |-- facility_c: string (nullable = true)
 |-- facility_id: string (nullable = true)
 |-- insurance_claim_status: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- primary_diagnosis_code: string (nullable = true)
 |-- procedure_: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

['facility_city', 'attending_staff_id', 'billing_amount', 'encounter_id', 'encounter_timestamp', 'encountertype', 'facility_c', 'facility_id', 'insurance_claim_status', 'patient_id', 'primary_diag

SynapseWidget(Synapse.DataFrame, 96c48dbf-0891-4771-919d-b2d073ea5c1e)

In [5]:
# CELL 5 — Validate Bronze clinical encounters ingestion

source_row_count = clinical_encounters_df.count()
bronze_row_count = bronze_clinical_encounters_df.count()

print("Source rows:", source_row_count)
print("Bronze rows:", bronze_row_count)

if source_row_count == bronze_row_count:
    print("Validation passed: all clinical encounter rows reached Bronze.")
else:
    print("Validation failed: row counts do not match.")

StatementMeta(, dc775bdb-7a22-4d21-9d5b-deb145de9eb1, 7, Finished, Available, Finished, False)

Source rows: 973589
Bronze rows: 973589
Validation passed: all clinical encounter rows reached Bronze.
